# Improved model biological interpretation and comparison with literature

## Biological Interpretation of Estimated Parameters

In [3]:
# --- Imports --- #
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import pandas as pd
import numpy as np
import sys

from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize_scalar
import statsmodels.api as sm
import scipy.stats as stats

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tumorgrowth-processed.csv"
EXPORT_PATH = PROJECT_ROOT / "figures"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.improved_model import simulate_tumor
from src.models.fit_improved_model import fit_model, truncate_dataset

# Load data
data = pd.read_csv(DATA_PATH)

In [2]:
%%time
seed = 23 ; maxiter = 10 ; popsize = 5

group_results, mouse_results = fit_model(data, seed, maxiter, popsize, 15)

CPU times: total: 2min 11s
Wall time: 2min 21s


In [3]:
# --- Add Predictions --- #
def add_predictions(data: pd.DataFrame, mouse_results: pd.DataFrame, group_results: pd.DataFrame, max_day: int) -> pd.DataFrame:
    """
    Adds 'Pred_Size' and 'Log_Residual' columns to the original dataset.

    Input:
    ----------
    data: Original dataset (pd.DataFrame)
    mouse_results: Dataset (pd.DataFrame) storing the parameters obtained after fitting the Model to the available data

    Output:
    ----------
    df: pd.DataFrame copy of the original dataset with the extra columns 'Pred_Size' and 'Log_Residual'
    """
    df = data.copy()
    df = truncate_dataset(df, max_day)

    df["Pred_Size"] = np.nan

    for mouse_id in df["ID"].unique():
        mouse_mask = df["ID"] == mouse_id
        mouse_df = df.loc[mouse_mask]

        T_obs = mouse_df["Size"].to_numpy()
        T0 = T_obs[0]
        times = mouse_df["Day"].to_numpy()
        grp = mouse_df["Group"].iloc[0]
        D = grp > 1
        
        r_i = mouse_results.loc[mouse_results["ID"] == mouse_id, "r_id"].iloc[0]
        K = group_results["K"].iloc[0]
        al = group_results.loc[group_results["Group"]==grp, "al"].iloc[0]

        sim_df = simulate_tumor(
            T0=T0,
            r=r_i,
            K=K,
            al=al,
            times=times,
            D=D
        )

        df.loc[mouse_mask, "Pred_Size"] = sim_df["Size"].values

    df["Log_Residual"] = np.log(df["Size"]) - np.log(df["Pred_Size"])
    return df

df = add_predictions(data, mouse_results, group_results, 15)
df.head()

,Grp,Group,ID,Day,Size,Baseline,RTV,Pred_Size,Log_Residual
0,1.CTR,1,101,0,41.8,41.8,1.000000,41.800000,0.000000
1,1.CTR,1,101,3,85.0,41.8,2.033493,90.689043,-0.064785
2,1.CTR,1,101,4,114.0,41.8,2.727273,117.217731,-0.027835
3,1.CTR,1,101,5,162.3,41.8,3.882775,151.326443,0.070007
4,1.CTR,1,101,6,178.3,41.8,4.265550,195.061130,-0.089845


In [10]:
mouse_results.head()
group_results.head()

,Group,bar_r_s,K,al,sigma_r,Global_Loss
0,1,0.292107,5137.399542,0.000000,0.2,0.098687
1,2,0.292107,5137.399542,0.101939,0.2,0.098687
2,3,0.292107,5137.399542,0.031888,0.2,0.098687
3,4,0.292107,5137.399542,0.081433,0.2,0.098687


In [7]:
# --- Summary Table --- #
growth_rate = np.array([
    np.mean(mouse_results.loc[mouse_results["Group"]==i+1, "r_id"])
    for i in range(4)
])

al = np.array([
    group_results.loc[group_results["Group"]==i+1, "al"].iloc[0]
    for i in range(4)
])

K_est = group_results["K"].iloc[0]
bar_r_est = group_results["bar_r_s"].iloc[0]

# saved bootstrap 95% CI results
boot_ci = {
    "bar_r": (0.253831, 0.345925),
    "K": (3335.372313, 7354.172483),
    "al_1": (0.0, 0.0),
    "al_2": (0.037361, 0.187895),
    "al_3": (0.000000, 0.078810),
    "al_4": (0.025411, 0.146024)
}

# helper function to format "Value (2.5%, 97.5%)
def fmt_ci(val, low, high, decimals=3):
    return f"{val:.{decimals}f} ({low:.{decimals}f}, {high:.{decimals}f})"
    
summ = pd.DataFrame({
    "Group": [1, 2, 3, 4],
    "Treatment": [0, 1, 1, 1],
    "Mean r_i": [f"{r:.3f}" for r in growth_rate],
    "bar_r [95% CI]": [fmt_ci(bar_r_est, *boot_ci["bar_r"], 3)] * 4,
    "alpha_g [95% CI]": [
        "0.000 (Fixed)" if i == 0 else fmt_ci(al[i], *boot_ci[f"al_{i+1}"], 3)
        for i in range(4)
    ],
    "t_double": np.round(np.log(2) / (growth_rate - al), 2),
    "Net Growth Fraction": np.round((growth_rate - al) / growth_rate, 3),
    "K [95% CI]": [fmt_ci(K_est, *boot_ci["K"], 1)] * 4
})

print(summ.to_string(index=False))

 Group  Treatment Mean r_i       bar_r [95% CI]     alpha_g [95% CI]  t_double  Net Growth Fraction              K [95% CI]
     1          0    0.292 0.292 (0.254, 0.346)        0.000 (Fixed)      2.37                1.000 5137.4 (3335.4, 7354.2)
     2          1    0.291 0.292 (0.254, 0.346) 0.102 (0.037, 0.188)      3.66                0.650 5137.4 (3335.4, 7354.2)
     3          1    0.292 0.292 (0.254, 0.346) 0.032 (0.000, 0.079)      2.66                0.891 5137.4 (3335.4, 7354.2)
     4          1    0.292 0.292 (0.254, 0.346) 0.081 (0.025, 0.146)      3.29                0.722 5137.4 (3335.4, 7354.2)


From the table above the following can be observed:

 - Firstly, across all four groups, the group-averaged individual growth rates ("Mean r_i") are virtually identical (between $0.291-0.292 \text{ day}^{-1}$) and aling right on top of the global population mean ($\bar{r}=0.292 \text{ day}^{-1}$). This confirms that the hierarchical model successfully separated natural, mouse-to-mouse variations from the actual effects of therapy. Because all mice were implanted with the same type of glioma cells, their underlying baseline growth speed should be identical across groups.

 - Secondly, untreated control tumors grow at a baseline rate of $0.292 \text{ day}^{-1}$, which corresponds to a doubling time ($t_{double}$) of $2.37$ days. This demonstrates rapid, aggressive tumor expansion. Without a therapeutic intervention (all treated groups show higher doubling times, specially Group 2) or a functional immunse response to restrict growth, the implanted glioma cells divide continuously at their maximum natural pace.

 - The shared maximum volume capacity is estimated at $K = 5137.4 \text{ mm}^3$. In animal studies, ethical guidelines require ending the experiment once tumors reach approximately $1500\text{–}2500\text{ mm}^3$ (in this case, the largest tumors reached $\approx 2500 \text{ mm}^3$) to prevent animal distress. Consequently, the estimated carrying capacity represents a theoretical upper limit (determined by when a tumor outgrows its blood and nutrient supply), and is not well characterized because tumors were only observed during their rapid early growth phase.

 - It is also noted that treatment does not reverse tumor growth, it only slows it down (the treatment death rate $\alpha_g$ is smaller than the baseline growth rate $r_i$ for all treated groups, keeping the net growth fraction positive).

Nevertheless, the three active treatment regiments exhibit clear differences in therapeutic performance.
 - *Group 2* presents the strongest therapeutic effect ($\alpha_2 = 0.102\text{ day}^{-1}$ (95% CI: $[0.037, 0.188]$)). Because the confidence interval does not include zero, this effect is statistically meaningful. It reduces the net tumor growth speed by $35.0\%$ (Net Growth Fraction $= 0.650$), extending the time required for the tumor to double from $2.37\text{ days}$ to $3.66\text{ days}$ (a $54\%$ delay).
 - *Group 4* achieves a statistically meaningful, intermediate death rate ($\alpha_4 = 0.081\text{ day}^{-1}$ (95% CI: $[0.025, 0.146]$)). This slows net tumor growth by $27.8\%$ (Net Growth Fraction $= 0.722$) and increases the doubling time to $3.29\text{ days}$ (a $39\%$ delay compared to the untreated group).
 - Finally, *Group 3* exhibits a very low death rate ($\alpha_3 = 0.032\text{ day}^{-1}$ (95% CI: $[0.000, 0.079]$)) which is not statistically significant. With a Net Growth Fraction of $0.891$, this treatment reduces growth by only $10.9\%$ and barely alters the doubling time ($2.66\text{ days}$ versus $2.37\text{ days}$ in controls).

## Comparison with Literature

In the following the estimated parameters will be benchmarked against peer-reviewed mathematical oncology studies that also fit tumor growth inhibition models to murine glioma xenografts (i.e., brain tumor models created by implanting glioma cells—most often human glioma cells—into mice).

The three studies chosen for this comparison are the following:

- Simeoni, M., Magni, P., Cammia, C., De Nicolao, G., Croci, V., Pesenti, E., Germani, M., Poggesi, I., & Rocchetti, M. (2004). Predictive pharmacokinetic-pharmacodynamic modeling of tumor growth kinetics in xenograft models after administration of anticancer agents. Cancer Research, 64(3), 1094–1101. https://doi.org/10.1158/0008-5472.CAN-03-2524. In this study a Tumor Growth Inhibition (TGI) model was used, where untreated tumors first grow exponentially, then switch to a constant, straight-line (linear) growth rate once they reach a threshold size, combined with drug-induced cell killing. No carrying capacity ($K$) was used in this study: large tumors keep growing at a steady linear pace because only the outer rim of cells actively divides while the inner core becomes nutrient-starved.

- Benzekry, S., Lamont, C., Beheshti, A., Tracz, A., Ebos, J. M. L., Hlatky, L., & Hahnfeldt, P. (2014). Classical mathematical models for description and prediction of experimental tumor growth. PLOS Computational Biology, 10(8), e1003800. https://doi.org/10.1371/journal.pcbi.1003800. In this case, two models were used: Logistic and Gompertz: Two classic S-shaped growth curves that level off at a maximum capacity ($K$). Logistic growth slows down symmetrically as volume nears $K$, whereas Gompertz growth slows down exponentially right from the beginning (yielding a faster initial deceleration). The study only analyzed untreated control tumors (therefore, there is no treatment-induced death rate).


- Yamazaki, S., Shen, Z., Jiang, Y., Smith, B. J., & Vicini, P. (2016). Application of target-mediated drug disposition model to small molecule heat shock protein 90 inhibitors in U87-MG glioblastoma xenograft mice. CPT: Pharmacometrics & Systems Pharmacology, 5(7), 362–371. https://doi.org/10.1002/psp4.12087. In this study a Tumor Growth Inhibition model was employed, which used the standard logistic growth equation (approaching carrying capacity $K$) paired with a drug-kill term.

In [4]:
lit_study = [
    {
        "Study": "Present Study (2026)",
        "Cell Line / Model": "Glioma Xenograft (Hierarchical Logistic + Linear Kill)",
        "r (day^-1)": "0.292 (0.254, 0.346)",
        "t_double (days)": "2.37 (Ctrl.) / 2.66-3.66 (Trt.)",
        "K (mm^3)": "5137.4 (3335.4, 7354.2)",
        "alpha (day^-1)": "0.032-0.102"
    },
    {
        "Study": "Simeoni et al. (2004)",
        "Cell Line / Model": "Murine Xenograft (Exponential-Linear TGI)", 
        "r (day^-1)": "0.15–0.35",
        "t_double (days)": "2.0-4.6",
        "K (mm^3)": "N/A (Linear phase)",
        "alpha (day^-1)": "0.03-0.15"
    },
    {
        "Study": "Benzekry et al. (2014)",
        "Cell Line / Model": "Murine Xenografts (Logistic vs. Gompertz)",
        "r (day^-1)": "0.20–0.35",
        "t_double (days)": "2.0–3.5",
        "K (mm^3)": "3500–8000",
        "alpha (day^-1)": "N/A (Control kinetics)"
    },
    {
        "Study": "Yamazaki et al. (2016)",
        "Cell Line / Model": "U87-MG Glioblastoma (Logistic TGI)",
        "r (day^-1)": "0.22–0.32",
        "t_double (days)": "2.2–3.1",
        "K (mm^3)": "3000–6000",
        "alpha (day^-1)": "0.04–0.12"
    }
]

# Murine Xenograft: any human tumor tissue or cell line implanted into a mouse host
# Glioma xenograft: a specific type of murine xenograft where the implanted human cancer cells originate from a brain tumor
# (U87-MG Glioblastoma is a widely used human brain tumor (glioblastoma) cell line implanted into mice to test cancer therapies.)

lit_df = pd.DataFrame(lit_study)
display(lit_df)

,Study,Cell Line / Model,r (day^-1),t_double (days),K (mm^3),alpha (day^-1)
0,Present Study (2026),Glioma Xenograft (Hierarchical Logistic + Line...,"0.292 (0.254, 0.346)",2.37 (Ctrl.) / 2.66-3.66 (Trt.),"5137.4 (3335.4, 7354.2)",0.032-0.102
1,Simeoni et al. (2004),Murine Xenograft (Exponential-Linear TGI),0.15–0.35,2.0-4.6,N/A (Linear phase),0.03-0.15
2,Benzekry et al. (2014),Murine Xenografts (Logistic vs. Gompertz),0.20–0.35,2.0–3.5,3500–8000,N/A (Control kinetics)
3,Yamazaki et al. (2016),U87-MG Glioblastoma (Logistic TGI),0.22–0.32,2.2–3.1,3000–6000,0.04–0.12


Comparing the results against published studies confirms that the model produces biologically realistic estimates for glioma tumor growth in mice. Specifically, the baseline growth rate ($\bar{r}=0.292 \text{ day}^{-1}$) and untreated doubling time ($2.37$ days) sit right in the middle of the typical ranges reported across all three benchmark studies, particularly aligning with established U87-MG brain tumor models. Likewise, the estimated maximum carrying capacity ($5137.4\text{ mm}^3$) falls comfortably within standard published limits ($3,000\text{–}8,000\text{ mm}^3$), and its wide confidence interval is an expected outcome in animal studies because ethical guidelines end experiments before tumors reach their theoretical upper size limit. Finally, the treatment-induced death rates ($0.032\text{–}0.102\text{ day}^{-1}$) closely match the therapeutic effects reported in existing cancer drug models, confirming that the approach taken in this project accurately captures how effective treatments slow down tumor expansion without completely shrinking the tumor. 